Mortgage document classification and extraction lab, updated for four synthetic sample PDFs.
*Co-authored with CoCo*

# Mortgage Document Classification and Extraction

*A hands-on lab with Snowflake Cortex AI Functions*

## Overview
Duration: ~25 minutes

Classify four mortgage-document PDFs with `AI_CLASSIFY`, extract business fields from the supported page types with `AI_EXTRACT`, and review the results.

The greenhouse gas report workflow is in a separate notebook: [Greenhouse reports notebook](https://github.com/Snowflake-Labs/sfquickstarts/blob/master/site/sfguides/src/document-intelligence-ai-functions/assets/vhol_document_ai_greenhouse_reports.ipynb).

### Prerequisites
- Familiarity with SQL and Snowflake stages
- A [Snowflake account](https://signup.snowflake.com/) with access to [Cortex AI Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql) and approved [cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)
- A role that can create a database, schema, stages, and tables, and a running warehouse
- The [mortgage sample PDFs](https://github.com/Snowflake-Labs/sfquickstarts/blob/master/site/sfguides/src/document-intelligence-ai-functions/assets/mortgage_reports.zip)

### You will build
- One result row per PDF with its document type, extracted fields, processing status, and review reason

## Setup
Duration: 5 minutes

Run the first cell to set the lab context. Then create the stage, upload the sample PDFs, and continue with the remaining cells in order.

In [ ]:
%%sql -r mtg_setup
CREATE DATABASE IF NOT EXISTS VHOL_DOCUMENT_AI_DB;
CREATE SCHEMA IF NOT EXISTS VHOL_DOCUMENT_AI_DB.DOCUMENT_AI;

USE DATABASE VHOL_DOCUMENT_AI_DB;
USE SCHEMA DOCUMENT_AI;

SELECT CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE();

### Create the Stage

Create an internal stage for the mortgage documents. Directory tables let the workflow discover the uploaded files. Use `SNOWFLAKE_SSE` encryption for the direct-file AI calls; see [CREATE STAGE](https://docs.snowflake.com/en/sql-reference/sql/create-stage) and [AI_CLASSIFY limitations](https://docs.snowflake.com/en/sql-reference/functions/ai_classify#limitations).

In [ ]:
%%sql -r mtg_stage
CREATE STAGE IF NOT EXISTS MORTGAGE_DOCS
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
    DIRECTORY = (ENABLE = TRUE);

### Upload the Sample PDFs

1. Download [mortgage_reports.zip](https://github.com/Snowflake-Labs/sfquickstarts/blob/master/site/sfguides/src/document-intelligence-ai-functions/assets/mortgage_reports.zip) and unzip it. Upload the individual PDFs, not the ZIP archive. For GitHub-hosted files, select **Download raw file**.
2. In Snowsight, go to **Ingestion > Add Data > Load files into a Stage**.
3. Select all four PDFs below, then choose **VHOL_DOCUMENT_AI_DB > DOCUMENT_AI > MORTGAGE_DOCS** and upload them.

| File | Document | Expected route |
|---|---|---|
| `01.pdf` | Declaration of Citizenship / Household Status | Citizenship declaration |
| `02.pdf` | Affidavit of Non-Taxpayer / Household Schedule | Non taxpayer affidavit |
| `03.pdf` | Residential Loan Application, pages 1 and 2 | Assets and transaction details |
| `04.pdf` | Settlement Charges / Disbursement Worksheet | Settlement charges |

Keep the filenames unchanged. The four PDFs are synthetic test documents with fictional names, numbers, and signatures.

See [Staging files using Snowsight](https://docs.snowflake.com/en/user-guide/data-load-local-file-system-stage-ui) for the upload workflow. Then refresh the directory table and confirm the files are visible.

In [ ]:
%%sql -r mtg_refresh
ALTER STAGE MORTGAGE_DOCS REFRESH;

LIST @MORTGAGE_DOCS;

### Select the PDFs

The query keeps every PDF on the stage. Each file is classified, and the files that do not match a route are reported as `Other`.

In [ ]:
%%sql -r mtg_files
CREATE OR REPLACE TEMP TABLE MORTGAGE_FILES AS
SELECT
    relative_path,
    SPLIT_PART(relative_path, '/', -1) AS filename,
    size AS file_size_bytes,
    last_modified
FROM DIRECTORY(@MORTGAGE_DOCS)
WHERE relative_path ILIKE '%.pdf';

SELECT * FROM MORTGAGE_FILES ORDER BY relative_path;

## Classify the Mortgage Documents
Duration: 7 minutes

Define one route for each supported page type. The `Other` route has no extraction schema, so those documents go to review without an `AI_EXTRACT` call.

### Define the Routes

In [ ]:
%%sql -r mtg_routes
CREATE OR REPLACE TEMP TABLE MORTGAGE_ROUTES (
    document_type VARCHAR,
    category_description VARCHAR,
    response_format VARIANT
);

INSERT INTO MORTGAGE_ROUTES
SELECT
    'Non taxpayer affidavit',
    'Affidavit of Non-Taxpayer with selected people, roles, years not filed, county, and notarization.',
    PARSE_JSON($${
  "county": "What county is entered near the top of the affidavit? Return an empty string if blank.",
  "affidavit_date": "What date does the notary acknowledgment in Section F give? Return YYYY-MM-DD, or an empty string if blank. Do not use the commission expiration date.",
  "commission_expiration_date": "What date is written as Commission expiration in the notary section? The date is in MM/DD/YYYY format. Return YYYY-MM-DD, or an empty string if blank."
}$$)

UNION ALL
SELECT
    'Assets and transaction details',
    'Residential loan application with Section VII assets and liabilities and Section VIII Details of Transaction, including purchase price, loan proceeds, and cash to close.',
    PARSE_JSON($${
  "total_assets": "What amount is written for TOTAL ASSETS in Section VII? Do not use Liquid Assets Subtotal. Return the written amount, or an empty string if blank.",
  "total_liabilities": "What amount is written for TOTAL LIABILITIES in Section VII, in the Unpaid balance column? Return the written amount, or an empty string if blank.",
  "purchase_price": "What amount is written for line a, Purchase price, in Section VIII, Details of Transaction? Return the written amount, or an empty string if blank.",
  "loan_proceeds": "What amount is written for line d, Loan proceeds, in Section VIII? Read the written amount; do not calculate it. Return an empty string if blank.",
  "cash_to_close": "What amount is written for line g, Cash to close, in Section VIII? Read the written amount; do not calculate it. Return an empty string if blank."
}$$)
UNION ALL
SELECT
    'Settlement charges',
    'Settlement charges worksheet listing fees by line number and line 1400 borrower and seller settlement charge totals.',
    PARSE_JSON($${
  "borrower_settlement_charges": "What amount is written on line 1400, Total Settlement Charges, in the Borrower paid column? Read the written total; do not calculate it. Return an empty string if blank.",
  "seller_settlement_charges": "What amount is written on line 1400, Total Settlement Charges, in the Seller paid column? Read the written total; do not calculate it. Return an empty string if blank.",
  "appraisal_fee": "What appraisal fee is written on line 803? Return the amount shown, or an empty string if blank.",
  "credit_report_fee": "What credit report fee is written on line 804? Return the amount shown, or an empty string if blank."
}$$)
UNION ALL
SELECT
    'Citizenship declaration',
    'Declaration of Citizenship / Household Status form with numbered blocks 1, 2, and 3 in Section II for citizenship status, followed by a signature and date.',
    PARSE_JSON($${
  "declarant_name": "What name is printed after I, at the top of the declaration? Return the printed name as written, or an empty string if blank.",
  "citizenship_status": "Which status block is checked? Return exactly one of: U.S. citizen; Noncitizen with eligible immigration status; Not contending eligible immigration status. Return an empty string if no block is checked.",
  "signature_date": "What date is written next to the signature? The date on this form is in MM/DD/YYYY format. Return YYYY-MM-DD, or an empty string if blank."
}$$)
UNION ALL
SELECT
    'Other',
    'Any other document, including notices and disclosures such as the Right to Financial Privacy Act notice or VA and HUD/FHA borrower notices, government program agreement summaries, and property insurance rating or underwriting forms. Choose Other for notice or disclosure text even when it appears alongside a loan application.',
    NULL::VARIANT;

### Classify Each PDF

Build the category list from the routes and classify each PDF. The final `TRUE` returns error details in the result instead of failing the query. Compare the assigned types with the document headings. `01.pdf` should route to `Citizenship declaration`, `02.pdf` to `Non taxpayer affidavit`, `03.pdf` to `Assets and transaction details`, and `04.pdf` to `Settlement charges`. The filename is not supplied to the classifier as evidence.

In [ ]:
%%sql -r mtg_classify
CREATE OR REPLACE TEMP TABLE MORTGAGE_CLASSIFIED AS
WITH categories AS (
    SELECT ARRAY_AGG(
        OBJECT_CONSTRUCT('label', document_type, 'description', category_description)
    ) WITHIN GROUP (ORDER BY document_type) AS categories
    FROM MORTGAGE_ROUTES
),
classified AS (
    SELECT
        f.relative_path,
        f.filename,
        AI_CLASSIFY(
            TO_FILE('@MORTGAGE_DOCS', f.relative_path),
            c.categories,
            {
                'task_description': 'Classify the supplied document page by its visible headings and sections. Distinguish loan application pages by section content. Use Other when none of the defined form types matches. Do not use the filename as evidence.',
                'output_mode': 'single'
            },
            TRUE  -- returns error details instead of failing the query
        ) AS raw_classification
    FROM MORTGAGE_FILES AS f
    CROSS JOIN categories AS c
)
SELECT
    relative_path,
    filename,
    NULLIF(raw_classification:value:labels[0]::VARCHAR, 'null') AS document_type,
    NULLIF(NULLIF(TRIM(raw_classification:error::VARCHAR), ''), 'null') AS classification_error
FROM classified;

SELECT filename, document_type, classification_error
FROM MORTGAGE_CLASSIFIED
ORDER BY filename;

## Extract Business Fields
Duration: 6 minutes

Route each classified document to its extraction schema and call `AI_EXTRACT` on the original PDF. Unsupported types and classification errors are skipped.

In [ ]:
%%sql -r mtg_extract
CREATE OR REPLACE TEMP TABLE MORTGAGE_EXTRACTED AS
WITH extraction_raw AS (
    SELECT
        c.relative_path,
        c.filename,
        c.document_type,
        AI_EXTRACT(
            file => TO_FILE('@MORTGAGE_DOCS', c.relative_path),
            responseFormat => r.response_format
        ) AS raw_extraction
    FROM MORTGAGE_CLASSIFIED AS c
    JOIN MORTGAGE_ROUTES AS r ON c.document_type = r.document_type
    WHERE c.classification_error IS NULL
      AND r.response_format IS NOT NULL
)
SELECT
    relative_path,
    filename,
    document_type,
    raw_extraction:response AS extracted_data,
    NULLIF(NULLIF(TRIM(raw_extraction:error::VARCHAR), ''), 'null') AS extraction_error
FROM extraction_raw;

SELECT filename, document_type, extracted_data, extraction_error
FROM MORTGAGE_EXTRACTED
ORDER BY filename;

## Review the Results
Duration: 4 minutes

Each document gets one processing status. A field is flagged for review when it is blank, null, or a placeholder such as `unknown`. This check does not verify that values are correct.

- `EXTRACTED`: the call succeeded and no field is blank, and any citizenship status is one of the three allowed options
- `FIELD_REVIEW`: at least one field is blank, or a citizenship status is outside the three allowed options
- `UNSUPPORTED_DOCUMENT`: no extraction schema exists for this type
- `CLASSIFICATION_ERROR` or `EXTRACTION_ERROR`: the call returned an error

In [ ]:
%%sql -r mtg_results
CREATE OR REPLACE TEMP TABLE MORTGAGE_RESULTS AS
WITH blank_fields AS (
    SELECT
        e.relative_path,
        LISTAGG(f.key, ', ') WITHIN GROUP (ORDER BY f.key) AS blank_fields
    FROM MORTGAGE_EXTRACTED AS e,
         LATERAL FLATTEN(INPUT => e.extracted_data) AS f
    WHERE NULLIF(TRIM(f.value::VARCHAR), '') IS NULL
       OR LOWER(TRIM(f.value::VARCHAR)) IN ('null', 'none', 'unknown', 'n/a')
    GROUP BY e.relative_path
)
SELECT
    c.filename,
    c.document_type,
    CASE
        WHEN c.classification_error IS NOT NULL THEN 'CLASSIFICATION_ERROR'
        WHEN r.response_format IS NULL THEN 'UNSUPPORTED_DOCUMENT'
        WHEN e.extraction_error IS NOT NULL OR e.extracted_data IS NULL THEN 'EXTRACTION_ERROR'
        WHEN b.blank_fields IS NOT NULL THEN 'FIELD_REVIEW'
        WHEN c.document_type = 'Citizenship declaration'
             AND NOT COALESCE(
                 (e.extracted_data:citizenship_status::VARCHAR) IN (
                     'U.S. citizen',
                     'Noncitizen with eligible immigration status',
                     'Not contending eligible immigration status'
                 ), FALSE)
            THEN 'FIELD_REVIEW'
        ELSE 'EXTRACTED'
    END AS processing_status,
    b.blank_fields,
    e.extracted_data
FROM MORTGAGE_CLASSIFIED AS c
LEFT JOIN MORTGAGE_ROUTES AS r ON c.document_type = r.document_type
LEFT JOIN MORTGAGE_EXTRACTED AS e ON c.relative_path = e.relative_path
LEFT JOIN blank_fields AS b ON c.relative_path = b.relative_path;

SELECT filename, document_type, processing_status, blank_fields, extracted_data
FROM MORTGAGE_RESULTS
ORDER BY processing_status, filename;

### Check Values Against the Source PDFs

The query below returns the extracted values as text. Compare them with the source pages. The table lists the values printed on the sample PDFs.

| Source | Field | Source value |
|---|---|---:|
| `03.pdf` | Section VII total assets | 355,000.00 |
| `03.pdf` | Section VII total liabilities, unpaid balance | 27,950.00 |
| `03.pdf` | Section VIII line a, purchase price | 480,000.00 |
| `03.pdf` | Section VIII line d, loan proceeds | 384,000.00 |
| `03.pdf` | Section VIII line g, cash to close | 96,963.68 |
| `04.pdf` | Line 1400, borrower paid | 11,963.68 |
| `04.pdf` | Line 1400, seller paid | 33,275.00 |

In [ ]:
%%sql -r mtg_compare
SELECT
    filename,
    extracted_data:total_assets::VARCHAR AS total_assets,
    extracted_data:total_liabilities::VARCHAR AS total_liabilities,
    extracted_data:purchase_price::VARCHAR AS purchase_price,
    extracted_data:loan_proceeds::VARCHAR AS loan_proceeds,
    extracted_data:cash_to_close::VARCHAR AS cash_to_close
FROM MORTGAGE_RESULTS
WHERE filename IN ('03.pdf');

In [ ]:
%%sql -r dataframe_1
SELECT
    filename,
    extracted_data:borrower_settlement_charges::VARCHAR AS borrower_settlement_charges,
    extracted_data:seller_settlement_charges::VARCHAR AS seller_settlement_charges
FROM MORTGAGE_RESULTS
WHERE filename IN ('04.pdf');

`EXTRACTED` means the call succeeded and no field is blank. It does not establish accuracy, completeness, or borrower eligibility. Check sample values against the source pages.

## Optional: Try It

These exercises re-run model calls, which can incur additional usage.

- Refine the description for the assets-and-transaction route, then re-run the classification cell and compare the results.
- Add one field to a response schema, re-run the extraction cell, and check the new value against its source.

## Cleanup

Both lab notebooks create their objects in `VHOL_DOCUMENT_AI_DB`. Dropping the database removes everything this notebook created, including the stage and uploaded PDFs.

Do not run this against a shared event database. In a shared account, ask the host which objects you can remove.

In [ ]:
%%sql -r mtg_cleanup
DROP DATABASE IF EXISTS VHOL_DOCUMENT_AI_DB;

## Conclusion

You classified mortgage pages, extracted fields from the supported types, and reviewed each document's status. The greenhouse reports notebook covers AGENTIC parsing and Cortex Search.